In [1]:
import requests
import json

In [2]:
API_TOKEN = "v5xoM3jMNGosH2sDN7XyAo7TCjIhqKOX8Ipaa9wR" #my token
DATA_CENTER = "fra1"  # our data center
SURVEY_ID = "SV_0j6lENWxftQXV2e"  # our survey_id

In [3]:
# Qualtrics API Base URLs
BASE_URL = f"https://{DATA_CENTER}.qualtrics.com/API/v3/survey-definitions/{SURVEY_ID}"
SURVEY_FLOW_URL = f"{BASE_URL}/flow"

In [4]:
# headers
HEADERS = {
    "Content-Type": "application/json",
    "Accept": "application/json",
    "X-API-TOKEN": API_TOKEN
}

### 1. Create a list with 100 articles (using Guardian data)

In [5]:
import pandas as pd

In [6]:
df = pd.read_csv("newsapi_guardian_20230220.csv")

In [7]:
#extract the first 100 rows, extract each row to the first 100 words.
articles_list = df["body"].dropna().astype(str).apply(lambda x: " ".join(x.split()[:100])).tolist()[:100]

In [8]:
print(articles_list[:5])

['Garcia Luna took millions in bribes from Sinaloa gang, Brooklyn corruption trial told A former Mexican law enforcement official once in charge of the fight against drug traffickers has been convicted at a US corruption trial over his ties to the Sinaloa cartel. Federal prosecutors in Brooklyn said Genaro Garcia Luna accepted millions of dollars in bribes from the cartel once run by Joaquin "El Chapo" Guzman in exchange for protection from arrest, safe passage for cocaine shipments and tipoffs about forthcoming law enforcement operations. Garcia Luna is one of the highest-ranking Mexican officials ever accused of ties to drug', "Reporters need protection from violence and intimidation - not a law that makes it easier to sack them During my first week as a journalist in a television station in Papua New Guinea, I got told by a bureaucrat in a large government department that I should be careful what I write about him as he knew the country's prime minister. That was 10 years ago. Since

In [9]:
len(articles_list)

100

### 2. Create blocks with questions

In [10]:
import time

In [11]:
def create_block(block_name):
    """Create a block and return the Block ID."""
    block_payload = {
        "Description": block_name,
        "Type": "Standard"
    }

    block_url = f"{BASE_URL}/blocks"
    response = requests.post(block_url, headers=HEADERS, json=block_payload)

    if response.status_code == 200:
        block_id = response.json()["result"]["BlockID"]
        print(f"Successfully created Block: {block_id}")
        return block_id
    else:
        print(f"Failed to create Block: {response.text}")
        return None

In [12]:
# Step 1: Create 100 Empty Blocks
block_ids = []
for i in range(1, 101):
    block_name = f"Article Block {i}"
    block_id = create_block(block_name)
    if block_id:
        block_ids.append(block_id)
    time.sleep(1)  # Avoid API Rate Limit

Successfully created Block: BL_03ccjjBzga05wIS
Successfully created Block: BL_1TzPVk83sA4xsIC
Successfully created Block: BL_3axvyEjS0t7LUCa
Successfully created Block: BL_8tVp5ZZhwwVBcqi
Successfully created Block: BL_3wRjISCPLEUklkG
Successfully created Block: BL_6EcEKvOHvcl4Xrg
Successfully created Block: BL_bEJKwKeWZZDe9b8
Successfully created Block: BL_3t5i0SC57k8exWm
Successfully created Block: BL_8pJaNvELtsjrPRY
Successfully created Block: BL_7R2EU1RLldsLCei
Successfully created Block: BL_ehsulABTAs0sqF0
Successfully created Block: BL_9NwqXNlHDtwMcpo
Successfully created Block: BL_eEeh74VrbMEqU6O
Successfully created Block: BL_2t8eb0jIbqk7yCO
Successfully created Block: BL_094reApRlAErOnQ
Successfully created Block: BL_2f4UvBgmXSh6xRY
Successfully created Block: BL_9HaR2FIOK1wHqoS
Successfully created Block: BL_5p3nRBm0VxLLM4C
Successfully created Block: BL_9uyXA5kMPzhQEgC
Successfully created Block: BL_6tGnXLIOeKWpieO
Successfully created Block: BL_2tLPKYy91yb3jp4
Successfully 

### 3. Create article question in each block

In [13]:
def create_question_article(article_text, block_id):
    """Create a question and assign it directly to the given Block ID."""
    question_payload = {
        "QuestionText": article_text,
        "DataExportTag": "ArticleQuestion",
        "QuestionType": "DB", 
        "Selector": "TB", 
        #"SubSelector": "",
        "Configuration": {
            "QuestionDescriptionOption": "UseText"
        }
    }

    question_url = f"{BASE_URL}/questions?blockId={block_id}"  # assign Block ID
    response = requests.post(question_url, headers=HEADERS, json=question_payload)

    if response.status_code == 200:
        question_id = response.json()["result"]["QuestionID"]
        print(f"Successfully created Question {question_id} in Block {block_id}")
        return question_id
    else:
        print(f"Failed to create Question: {response.text}")
        return None

In [14]:
# Step 2: for each block，add article first
article_question_ids = []

for i, (article, block_id) in enumerate(zip(articles_list, block_ids)):
    print(f"\nProcessing Block {i+1} - {block_id}")
    article_qid = create_question_article(article, block_id)
    if article_qid:
        article_question_ids.append(article_qid)
    time.sleep(1)  # Avoid API Rate Limit


Processing Block 1 - BL_03ccjjBzga05wIS
Successfully created Question QID950 in Block BL_03ccjjBzga05wIS

Processing Block 2 - BL_1TzPVk83sA4xsIC
Successfully created Question QID951 in Block BL_1TzPVk83sA4xsIC

Processing Block 3 - BL_3axvyEjS0t7LUCa
Successfully created Question QID952 in Block BL_3axvyEjS0t7LUCa

Processing Block 4 - BL_8tVp5ZZhwwVBcqi
Successfully created Question QID953 in Block BL_8tVp5ZZhwwVBcqi

Processing Block 5 - BL_3wRjISCPLEUklkG
Successfully created Question QID954 in Block BL_3wRjISCPLEUklkG

Processing Block 6 - BL_6EcEKvOHvcl4Xrg
Successfully created Question QID955 in Block BL_6EcEKvOHvcl4Xrg

Processing Block 7 - BL_bEJKwKeWZZDe9b8
Successfully created Question QID956 in Block BL_bEJKwKeWZZDe9b8

Processing Block 8 - BL_3t5i0SC57k8exWm
Successfully created Question QID957 in Block BL_3t5i0SC57k8exWm

Processing Block 9 - BL_8pJaNvELtsjrPRY
Successfully created Question QID958 in Block BL_8pJaNvELtsjrPRY

Processing Block 10 - BL_7R2EU1RLldsLCei
Succ

### 4. Create the same follow-up questions in each block

In [15]:
def create_topic_question(block_id):
    """Create a Topic Question in a specific block following the official API format."""
    payload = {
        "QuestionType": "MC",
        "Selector": "SAVR",
        "SubSelector": "TX",
        "QuestionText": "According to you, which of the topics below best describes the content of this text?",
        "DataExportTag": "Topic",
        "ChoiceOrder": ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12"],
        "Choices": {
            "1": {"Display": "National politics"},
            "2": {"Display": "International news"},
            "3": {"Display": "Business or economics"},
            "4": {"Display": "Crime and security"},
            "5": {"Display": "Education"},
            "6": {"Display": "Environment or climate"},
            "7": {"Display": "Health and wellbeing"},
            "8": {"Display": "Science and technology"},
            "9": {"Display": "Lifestyle (sports, arts, culture)"},
            "10": {"Display": "Entertainment and celebrity"},
            "11": {"Display": "Funny news"},
            "12": {"Display": "I do not know"}
        },
        "Validation": {
            "Settings": {
                "ForceResponse": "ON",
                "ForceResponseType": "ON"
            }
        },
        "Configuration": {
            "QuestionDescriptionOption": "UseText",
            #"TextPosition": "inline",
            #"ChoiceColumnWidth": 25,
            #"RepeatHeaders": "none",
            #"WhiteSpace": "OFF",
            #"LabelPosition": "SIDE",
            #"NumColumns": 1,
            #"MobileFirst": True
        }
    }

    url = f"{BASE_URL}/questions?blockId={block_id}"
    response = requests.post(url, headers=HEADERS, json=payload)

    if response.status_code == 200:
        question_id = response.json()["result"]["QuestionID"]
        print(f"Successfully created Topic Question {question_id} in Block {block_id}")
        return question_id
    else:
        print(f"Failed to create Topic Question: {response.text}")
        return None

In [16]:
# Execute the creation of Topic Questions
for block_id in block_ids:
    create_topic_question(block_id)
    time.sleep(1)

Successfully created Topic Question QID1050 in Block BL_03ccjjBzga05wIS
Successfully created Topic Question QID1051 in Block BL_1TzPVk83sA4xsIC
Successfully created Topic Question QID1052 in Block BL_3axvyEjS0t7LUCa
Successfully created Topic Question QID1053 in Block BL_8tVp5ZZhwwVBcqi
Successfully created Topic Question QID1054 in Block BL_3wRjISCPLEUklkG
Successfully created Topic Question QID1055 in Block BL_6EcEKvOHvcl4Xrg
Successfully created Topic Question QID1056 in Block BL_bEJKwKeWZZDe9b8
Successfully created Topic Question QID1057 in Block BL_3t5i0SC57k8exWm
Successfully created Topic Question QID1058 in Block BL_8pJaNvELtsjrPRY
Successfully created Topic Question QID1059 in Block BL_7R2EU1RLldsLCei
Successfully created Topic Question QID1060 in Block BL_ehsulABTAs0sqF0
Successfully created Topic Question QID1061 in Block BL_9NwqXNlHDtwMcpo
Successfully created Topic Question QID1062 in Block BL_eEeh74VrbMEqU6O
Successfully created Topic Question QID1063 in Block BL_2t8eb0jI

In [17]:
def create_gender_question(block_id):
    """Create a 'Gender' question within a specific block."""
    gender_question_payload = {
        "DataExportTag": "Gender",
        "QuestionText": (
            "Please answer the question below for the first person in the text who is being interviewed."
            "<br><br>Of what gender is the person interviewed in the text?"
            "<br><i>Note: If no persons are featured in the text, select 'not applicable'."
            " Only if the gender of the person interviewed is unclear, select 'not disclosed'.<br></i><br>"
            "The person interviewed in the text is a:<br>"
        ),
        "QuestionType": "MC",
        "Selector": "SAVR",
        "SubSelector": "TX",
        "Configuration": {
            "QuestionDescriptionOption": "UseText"
        },
        "Validation": {
            "Settings": {
                "ForceResponse": "ON",
                "ForceResponseType": "ON"
            }
        },
        "Choices": {
            "1": {"Display": "Man"},
            "2": {"Display": "Woman"},
            "3": {"Display": "Not applicable"},
            "4": {"Display": "Not disclosed"}
        },
        "ChoiceOrder": ["1", "2", "3", "4"]
    }

    url = f"{BASE_URL}/questions?blockId={block_id}"
    response = requests.post(url, headers=HEADERS, json=gender_question_payload)

    if response.status_code == 200:
        question_id = response.json()["result"]["QuestionID"]
        print(f"Successfully created Gender Question {question_id} in Block {block_id}")
        return question_id
    else:
        print(f"Failed to create Gender Question: {response.text}")
        return None

In [18]:
for block_id in block_ids:
    create_gender_question(block_id)
    time.sleep(1)

Successfully created Gender Question QID1150 in Block BL_03ccjjBzga05wIS
Successfully created Gender Question QID1151 in Block BL_1TzPVk83sA4xsIC
Successfully created Gender Question QID1152 in Block BL_3axvyEjS0t7LUCa
Successfully created Gender Question QID1153 in Block BL_8tVp5ZZhwwVBcqi
Successfully created Gender Question QID1154 in Block BL_3wRjISCPLEUklkG
Successfully created Gender Question QID1155 in Block BL_6EcEKvOHvcl4Xrg
Successfully created Gender Question QID1156 in Block BL_bEJKwKeWZZDe9b8
Successfully created Gender Question QID1157 in Block BL_3t5i0SC57k8exWm
Successfully created Gender Question QID1158 in Block BL_8pJaNvELtsjrPRY
Successfully created Gender Question QID1159 in Block BL_7R2EU1RLldsLCei
Successfully created Gender Question QID1160 in Block BL_ehsulABTAs0sqF0
Successfully created Gender Question QID1161 in Block BL_9NwqXNlHDtwMcpo
Successfully created Gender Question QID1162 in Block BL_eEeh74VrbMEqU6O
Successfully created Gender Question QID1163 in Blo

In [19]:
def create_expertise_question(block_id):
    """Create the 'Expertise' question within a specific block."""
    question_payload = {
        "DataExportTag": "Expertise",
        "QuestionText": "Please answer the question below for the first person in the text who is being interviewed.<br><br>"
                        "How would you assess the expertise of the person interviewed in the text?<br>"
                        "<i>Note: If no persons are featured in the text, select 'not applicable'.<br></i><br>"
                        "The person interviewed in the text has:<br>",
        "QuestionType": "MC",
        "Selector": "SAVR",
        "SubSelector": "TX",
        "Configuration": {
            "QuestionDescriptionOption": "UseText",
            #"NumColumns": 7,
        },
        "Validation": {
            "Settings": {
                "ForceResponse": "ON",
                "ForceResponseType": "ON"
            }
        },
        "Choices": {
            "1": {"Display": "1. Low expertise"},
            "2": {"Display": "2"},
            "3": {"Display": "3"},
            "4": {"Display": "4"},
            "5": {"Display": "5. High expertise"}
        },
        "ChoiceOrder": ["1", "2", "3", "4", "5", "6", "7"]
    }

    url = f"{BASE_URL}/questions?blockId={block_id}"
    response = requests.post(url, headers=HEADERS, json=question_payload)

    if response.status_code == 200:
        question_id = response.json()["result"]["QuestionID"]
        print(f"Successfully created Gender Question {question_id} in Block {block_id}")
        return question_id
    else:
        print(f"Failed to create Gender Question: {response.text}")
        return None

In [20]:
for block_id in block_ids:
    create_expertise_question(block_id)
    time.sleep(1) 

Successfully created Gender Question QID1250 in Block BL_03ccjjBzga05wIS
Successfully created Gender Question QID1251 in Block BL_1TzPVk83sA4xsIC
Successfully created Gender Question QID1252 in Block BL_3axvyEjS0t7LUCa
Successfully created Gender Question QID1253 in Block BL_8tVp5ZZhwwVBcqi
Successfully created Gender Question QID1254 in Block BL_3wRjISCPLEUklkG
Successfully created Gender Question QID1255 in Block BL_6EcEKvOHvcl4Xrg
Successfully created Gender Question QID1256 in Block BL_bEJKwKeWZZDe9b8
Successfully created Gender Question QID1257 in Block BL_3t5i0SC57k8exWm
Successfully created Gender Question QID1258 in Block BL_8pJaNvELtsjrPRY
Successfully created Gender Question QID1259 in Block BL_7R2EU1RLldsLCei
Successfully created Gender Question QID1260 in Block BL_ehsulABTAs0sqF0
Successfully created Gender Question QID1261 in Block BL_9NwqXNlHDtwMcpo
Successfully created Gender Question QID1262 in Block BL_eEeh74VrbMEqU6O
Successfully created Gender Question QID1263 in Blo

In [21]:
def create_fatuality_question(block_id):
    """Create the 'Factuality' question within a specific block."""
    question_payload = {
        "DataExportTag": "Factuality",
        "QuestionText": "Overall, would you say that the text reports predominantly facts or opinions?",
        "QuestionType": "MC",
        "Selector": "SAVR",
        "SubSelector": "TX",
        "Configuration": {
            "QuestionDescriptionOption": "UseText",
        },
        "Validation": {
            "Settings": {
                "ForceResponse": "ON",
                "ForceResponseType": "ON"
            }
        },
        "Choices": {
            "1": {"Display": "1. Facts only"},
            "2": {"Display": "2"},
            "3": {"Display": "3"},
            "4": {"Display": "4"},
            "5": {"Display": "5. Opinions only"}
        },
        "ChoiceOrder": ["1", "2", "3", "4", "5"]
    }

    url = f"{BASE_URL}/questions?blockId={block_id}"
    response = requests.post(url, headers=HEADERS, json=question_payload)

    if response.status_code == 200:
        question_id = response.json()["result"]["QuestionID"]
        print(f"Successfully created Factuality Question {question_id} in Block {block_id}")
        return question_id
    else:
        print(f"Failed to create Factuality Question: {response.text}")
        return None

In [22]:
for block_id in block_ids:
    create_fatuality_question(block_id)
    time.sleep(1) 

Successfully created Factuality Question QID1350 in Block BL_03ccjjBzga05wIS
Successfully created Factuality Question QID1351 in Block BL_1TzPVk83sA4xsIC
Successfully created Factuality Question QID1352 in Block BL_3axvyEjS0t7LUCa
Successfully created Factuality Question QID1353 in Block BL_8tVp5ZZhwwVBcqi
Successfully created Factuality Question QID1354 in Block BL_3wRjISCPLEUklkG
Successfully created Factuality Question QID1355 in Block BL_6EcEKvOHvcl4Xrg
Successfully created Factuality Question QID1356 in Block BL_bEJKwKeWZZDe9b8
Successfully created Factuality Question QID1357 in Block BL_3t5i0SC57k8exWm
Successfully created Factuality Question QID1358 in Block BL_8pJaNvELtsjrPRY
Successfully created Factuality Question QID1359 in Block BL_7R2EU1RLldsLCei
Successfully created Factuality Question QID1360 in Block BL_ehsulABTAs0sqF0
Successfully created Factuality Question QID1361 in Block BL_9NwqXNlHDtwMcpo
Successfully created Factuality Question QID1362 in Block BL_eEeh74VrbMEqU6O

In [23]:
def create_humaninterest_question(block_id):
    """Create the 'HumanInterest' question within a specific block."""
    question_payload = {
        "DataExportTag": "HumanInterest",
        "QuestionText": "Does the text provide a human example or \"human face\" on the issue that it discusses?",
        "QuestionType": "MC",
        "Selector": "SAVR",
        "SubSelector": "TX",
        "Configuration": {
            "QuestionDescriptionOption": "UseText",
        },
        "Validation": {
            "Settings": {
                "ForceResponse": "ON",
                "ForceResponseType": "ON"
            }
        },
        "Choices": {
            "1": {"Display": "Yes"},
            "2": {"Display": "No"}
        },
        "ChoiceOrder": ["1", "2"]
    }

    url = f"{BASE_URL}/questions?blockId={block_id}"
    response = requests.post(url, headers=HEADERS, json=question_payload)

    if response.status_code == 200:
        question_id = response.json()["result"]["QuestionID"]
        print(f"Successfully created HumanInterest Question {question_id} in Block {block_id}")
        return question_id
    else:
        print(f"Failed to create HumanInterest Question: {response.text}")
        return None

In [24]:
for block_id in block_ids:
    create_humaninterest_question(block_id)
    time.sleep(1) 

Successfully created HumanInterest Question QID1450 in Block BL_03ccjjBzga05wIS
Successfully created HumanInterest Question QID1451 in Block BL_1TzPVk83sA4xsIC
Successfully created HumanInterest Question QID1452 in Block BL_3axvyEjS0t7LUCa
Successfully created HumanInterest Question QID1453 in Block BL_8tVp5ZZhwwVBcqi
Successfully created HumanInterest Question QID1454 in Block BL_3wRjISCPLEUklkG
Successfully created HumanInterest Question QID1455 in Block BL_6EcEKvOHvcl4Xrg
Successfully created HumanInterest Question QID1456 in Block BL_bEJKwKeWZZDe9b8
Successfully created HumanInterest Question QID1457 in Block BL_3t5i0SC57k8exWm
Successfully created HumanInterest Question QID1458 in Block BL_8pJaNvELtsjrPRY
Successfully created HumanInterest Question QID1459 in Block BL_7R2EU1RLldsLCei
Successfully created HumanInterest Question QID1460 in Block BL_ehsulABTAs0sqF0
Successfully created HumanInterest Question QID1461 in Block BL_9NwqXNlHDtwMcpo
Successfully created HumanInterest Quest

In [25]:
def create_sentiment_question(block_id):
    """Create the 'Sentiment' question within a specific block."""
    question_payload = {
        "DataExportTag": "Sentiment",
        "QuestionText": "How would you assess the tone of the text?",
        "QuestionType": "MC",
        "Selector": "SAVR",
        "SubSelector": "TX",
        "Configuration": {
            "QuestionDescriptionOption": "UseText",
        },
        "Validation": {
            "Settings": {
                "ForceResponse": "ON",
                "ForceResponseType": "ON"
            }
        },
        "Choices": {
            "1": {"Display": "1. Very negative"},
            "2": {"Display": "2"},
            "3": {"Display": "3"},
            "4": {"Display": "4"},
            "5": {"Display": "5. Very positive"}
        },
        "ChoiceOrder": ["1", "2", "3", "4", "5"]
    }

    url = f"{BASE_URL}/questions?blockId={block_id}"
    response = requests.post(url, headers=HEADERS, json=question_payload)

    if response.status_code == 200:
        question_id = response.json()["result"]["QuestionID"]
        print(f"Successfully created Sentiment Question {question_id} in Block {block_id}")
        return question_id
    else:
        print(f"Failed to create Sentiment Question: {response.text}")
        return None

In [26]:
for block_id in block_ids:
    create_sentiment_question(block_id)
    time.sleep(1) 

Successfully created Sentiment Question QID1550 in Block BL_03ccjjBzga05wIS
Successfully created Sentiment Question QID1551 in Block BL_1TzPVk83sA4xsIC
Successfully created Sentiment Question QID1552 in Block BL_3axvyEjS0t7LUCa
Successfully created Sentiment Question QID1553 in Block BL_8tVp5ZZhwwVBcqi
Successfully created Sentiment Question QID1554 in Block BL_3wRjISCPLEUklkG
Successfully created Sentiment Question QID1555 in Block BL_6EcEKvOHvcl4Xrg
Successfully created Sentiment Question QID1556 in Block BL_bEJKwKeWZZDe9b8
Successfully created Sentiment Question QID1557 in Block BL_3t5i0SC57k8exWm
Successfully created Sentiment Question QID1558 in Block BL_8pJaNvELtsjrPRY
Successfully created Sentiment Question QID1559 in Block BL_7R2EU1RLldsLCei
Successfully created Sentiment Question QID1560 in Block BL_ehsulABTAs0sqF0
Successfully created Sentiment Question QID1561 in Block BL_9NwqXNlHDtwMcpo
Successfully created Sentiment Question QID1562 in Block BL_eEeh74VrbMEqU6O
Successfully

In [27]:
def create_articleend_question(block_id):
    """Create the 'ArticleEnd' question within a specific block."""
    question_payload = {
        "QuestionText": "You have now completed the all questions for this article!\n\nPlease click next to answer the same questions about another article (displayed in random order).",
        "DataExportTag": "ArticleEnd",
        "QuestionType": "DB", 
        "Selector": "TB", 
        #"SubSelector": "",
        "Configuration": {
            "QuestionDescriptionOption": "UseText"
        }
    }

    url = f"{BASE_URL}/questions?blockId={block_id}"
    response = requests.post(url, headers=HEADERS, json=question_payload)

    if response.status_code == 200:
        question_id = response.json()["result"]["QuestionID"]
        print(f"Successfully created ArticleEnd Question {question_id} in Block {block_id}")
        return question_id
    else:
        print(f"Failed to create ArticleEnd Question: {response.text}")
        return None

In [28]:
for block_id in block_ids:
    create_articleend_question(block_id)
    time.sleep(1) 

Successfully created ArticleEnd Question QID1650 in Block BL_03ccjjBzga05wIS
Successfully created ArticleEnd Question QID1651 in Block BL_1TzPVk83sA4xsIC
Successfully created ArticleEnd Question QID1652 in Block BL_3axvyEjS0t7LUCa
Successfully created ArticleEnd Question QID1653 in Block BL_8tVp5ZZhwwVBcqi
Successfully created ArticleEnd Question QID1654 in Block BL_3wRjISCPLEUklkG
Successfully created ArticleEnd Question QID1655 in Block BL_6EcEKvOHvcl4Xrg
Successfully created ArticleEnd Question QID1656 in Block BL_bEJKwKeWZZDe9b8
Successfully created ArticleEnd Question QID1657 in Block BL_3t5i0SC57k8exWm
Successfully created ArticleEnd Question QID1658 in Block BL_8pJaNvELtsjrPRY
Successfully created ArticleEnd Question QID1659 in Block BL_7R2EU1RLldsLCei
Successfully created ArticleEnd Question QID1660 in Block BL_ehsulABTAs0sqF0
Successfully created ArticleEnd Question QID1661 in Block BL_9NwqXNlHDtwMcpo
Successfully created ArticleEnd Question QID1662 in Block BL_eEeh74VrbMEqU6O

### 4. Create a randomizer for blocks

In [29]:
res = requests.get(f'https://{DATA_CENTER}.qualtrics.com/API/v3/survey-definitions/{SURVEY_ID}/flow', headers=HEADERS)

In [30]:
current_flow = res.json()['result']

In [31]:
current_flow

{'Type': 'Root',
 'FlowID': 'FL_1',
 'Flow': [{'Type': 'Block',
   'ID': 'BL_0B4ka3RFifrgkfQ',
   'FlowID': 'FL_2',
   'Autofill': []},
  {'Type': 'Standard',
   'ID': 'BL_8B2R1OTHvELLa6y',
   'FlowID': 'FL_3',
   'Autofill': []},
  {'Type': 'Standard',
   'ID': 'BL_0ValuP3rzc9VWJw',
   'FlowID': 'FL_4',
   'Autofill': []},
  {'Type': 'Standard',
   'ID': 'BL_03ccjjBzga05wIS',
   'FlowID': 'FL_49',
   'Autofill': []},
  {'Type': 'Standard',
   'ID': 'BL_1TzPVk83sA4xsIC',
   'FlowID': 'FL_50',
   'Autofill': []},
  {'Type': 'Standard',
   'ID': 'BL_3axvyEjS0t7LUCa',
   'FlowID': 'FL_51',
   'Autofill': []},
  {'Type': 'Standard',
   'ID': 'BL_8tVp5ZZhwwVBcqi',
   'FlowID': 'FL_52',
   'Autofill': []},
  {'Type': 'Standard',
   'ID': 'BL_3wRjISCPLEUklkG',
   'FlowID': 'FL_53',
   'Autofill': []},
  {'Type': 'Standard',
   'ID': 'BL_6EcEKvOHvcl4Xrg',
   'FlowID': 'FL_54',
   'Autofill': []},
  {'Type': 'Standard',
   'ID': 'BL_bEJKwKeWZZDe9b8',
   'FlowID': 'FL_55',
   'Autofill': []},
  

In [32]:
new_flow = {
    'Type': 'Root',
    'FlowID': 'FL_1',
    'Flow': [
        {'Type': 'Block',
         'ID': 'BL_0B4ka3RFifrgkfQ',
         'FlowID': 'FL_2',
         'Autofill': []},
        {'Type': 'Standard',
         'ID': 'BL_8B2R1OTHvELLa6y',
         'FlowID': 'FL_3',
         'Autofill': []},
        {'Type': 'Standard',
         'ID': 'BL_0ValuP3rzc9VWJw',
         'FlowID': 'FL_5',
         'Autofill': []},
        {
            'Type': 'BlockRandomizer',
            'FlowID': 'FL_6',
            'SubSet': 1,
            'EvenPresentation': True,
            'Flow': [
                {'Type': 'Standard', 'ID': 'BL_03ccjjBzga05wIS', 'FlowID': 'FL_49', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_1TzPVk83sA4xsIC', 'FlowID': 'FL_50', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_3axvyEjS0t7LUCa', 'FlowID': 'FL_51', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_8tVp5ZZhwwVBcqi', 'FlowID': 'FL_52', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_3wRjISCPLEUklkG', 'FlowID': 'FL_53', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_6EcEKvOHvcl4Xrg', 'FlowID': 'FL_54', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_bEJKwKeWZZDe9b8', 'FlowID': 'FL_55', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_3t5i0SC57k8exWm', 'FlowID': 'FL_56', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_8pJaNvELtsjrPRY', 'FlowID': 'FL_57', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_7R2EU1RLldsLCei', 'FlowID': 'FL_58', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_ehsulABTAs0sqF0', 'FlowID': 'FL_59', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_9NwqXNlHDtwMcpo', 'FlowID': 'FL_60', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_eEeh74VrbMEqU6O', 'FlowID': 'FL_61', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_2t8eb0jIbqk7yCO', 'FlowID': 'FL_62', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_094reApRlAErOnQ', 'FlowID': 'FL_63', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_2f4UvBgmXSh6xRY', 'FlowID': 'FL_64', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_9HaR2FIOK1wHqoS', 'FlowID': 'FL_65', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_5p3nRBm0VxLLM4C', 'FlowID': 'FL_66', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_9uyXA5kMPzhQEgC', 'FlowID': 'FL_67', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_6tGnXLIOeKWpieO', 'FlowID': 'FL_68', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_2tLPKYy91yb3jp4', 'FlowID': 'FL_69', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_aXxWCEIxgx8uHwq', 'FlowID': 'FL_70', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_24cGpJOlMhd6e9g', 'FlowID': 'FL_71', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_bDFsdWIguY0yY2q', 'FlowID': 'FL_72', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_ehsGcpAs18E1xpI', 'FlowID': 'FL_73', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_aeD38kCN866FKJ0', 'FlowID': 'FL_74', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_a5h4GXzaRsbaj4i', 'FlowID': 'FL_75', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_eWObDk4imiZwHMq', 'FlowID': 'FL_76', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_3Q7AKJ9jafy3Hf0', 'FlowID': 'FL_77', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_bBGbWpqeVK0fRj0', 'FlowID': 'FL_78', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_b74j4XyxzcWnHpk', 'FlowID': 'FL_79', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_1zR9NGvqC51EJkW', 'FlowID': 'FL_80', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_dmyYXPVqbUu5Qd8', 'FlowID': 'FL_81', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_aXXiMfAsjMhwJNk', 'FlowID': 'FL_82', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_0IHkLs1XUqZzx42', 'FlowID': 'FL_83', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_08H8kvlMTgdi6KW', 'FlowID': 'FL_84', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_3vFPlhn4EqI5sI6', 'FlowID': 'FL_85', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_8w9LT5QVMjinECq', 'FlowID': 'FL_86', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_dcnfh0hz9w30bum', 'FlowID': 'FL_87', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_6o53lBJVDdlWblk', 'FlowID': 'FL_88', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_exqOyE7eCk9Qaqi', 'FlowID': 'FL_89', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_3vHESWxfK0mMhq6', 'FlowID': 'FL_90', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_39ONzgm0OPoLb0y', 'FlowID': 'FL_91', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_1MluTRR96934BwO', 'FlowID': 'FL_92', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_em5ocvlQaErAIIe', 'FlowID': 'FL_93', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_5A0Ur3NW1rZCMWG', 'FlowID': 'FL_94', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_af4ug2guWXJ0pO6', 'FlowID': 'FL_95', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_aNcF0rbhFBFw60h', 'FlowID': 'FL_96', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_2ExMLmMS9hp9TPo', 'FlowID': 'FL_97', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_7SHuqJcBKhHsZbS', 'FlowID': 'FL_98', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_9p8qxY4paXI3cXz', 'FlowID': 'FL_99', 'Autofill': []},
                {'Type': 'Standard', 'ID': 'BL_b9gXLzMK5pVNSy3', 'FlowID': 'FL_100', 'Autofill': []}
            ]
        },
        {'Type': 'Standard',
         'ID': 'BL_5gaYFcBhPZ5Rge3',
         'FlowID': 'FL_4',
         'Autofill': []}
    ]
}


In [33]:
### This is how you put things back into the API
requests.put(f'https://{DATA_CENTER}.qualtrics.com/API/v3/survey-definitions/{SURVEY_ID}/flow', headers=HEADERS, json=new_flow)

<Response [200]>